# TP 3 - Agent PydanticAI


---
## 0. Configuration partagée


In [1]:
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from shared.agent_utils import run_agent_realtime_logging
from shared.agent_tools import tool_get_current_date, tool_geocode_location, tool_get_weather, tool_search_nearby, tool_retrieve_docs, web_search, web_extract
from shared.config import ROOT_DIR, google_model_settings, project_settings

**TODO — Configuration partagée et périmètre de code**

Fichiers à modifier :
- `shared/agent_tools.py`
- `TP3_travel_planner_Agent/3_1_tooling_assistant.ipynb`

Ce TP introduit un agent qui s'appuie sur plusieurs outils externes et internes

`tool_get_current_date` : fonction qui convertit les dates relatives en dates calendrier explicites

`tool_geocode_location` : fonction qui transforme un lieu texte en coordonnées géographiques

`tool_get_weather` : fonction qui retourne une prévision météo sur une plage de dates

`tool_retrieve_docs` : fonction qui récupère les passages RAG internes les plus pertinents

`tool_search_nearby` : fonction qui cherche des lieux proches autour de coordonnées avec filtres

`web_search` : fonction qui lance une recherche web externe

`web_extract` : fonction qui lit le contenu d'une page web trouvée

`Place` : classe qui représente un lieu avec un nom et des coordonnées


In [2]:
# TODO : configurer le modèle et les settings partagés pour tous les use cases
model = GoogleModel(
    model_name=project_settings.llm_model_name,
    provider=GoogleProvider(api_key=project_settings.google_api_key),
)
settings = google_model_settings


In [3]:

base_system_prompt = """

# RÈGLES

Tu es un assistant de planification de voyage orienté RAG.
Tu dois produire des recommandations personnalisées à partir de faits récupérés par outils.

## Règles générales

### Usage des outils et du contexte
- Utiliser les outils avant de répondre dès qu'un fait est nécessaire.
- Chaîner les outils si besoin (date -> geocode -> météo, docs -> web_search -> web_extract, etc.).
- N'utiliser que des faits issus des outils et du contexte récupéré.
- Citer les preuves factuelles (ex: [tool_retrieve_docs:source], [web_search:url]).
- Si les données sont partielles ou contradictoires, le dire explicitement.
- Si une information manque, la lister clairement sans inventer.
- Ignorer le contexte hors sujet.

### Style de réponse
- Réponse concise et précise.
- Pas de Markdown décoratif.
- Pour chaque recommandation: 1 raison + 1 détail pratique (horaire, lieu, budget, logistique).
- Préférer des actions concrètes aux formulations vagues.

### Format attendu
1) Résumé: réponse directe
2) Plan: séquence concrète (jour par jour ou par objectif)
3) Budget: estimation par catégorie uniquement sur faits disponibles
4) Preuves: faits clés utilisés (sources/outils)
5) Informations manquantes: liste courte et explicite

Important: ne pas interrompre avec des questions en cours de route. Produire la meilleure réponse possible avec les données disponibles, puis lister les manques.

## Consignes spécifiques par outil

"""


---

## 1. Cas d'usage 1 - Rome en 4 jours

Construire une chaîne d'outils fiable pour un séjour court.
Ordre attendu: date -> géocodage -> météo -> docs -> réponse.
Vérification concrète dans la trace: chaque recommandation doit être justifiée par au moins un résultat d'outil et respecter durée/budget/préférences.

In [4]:
prompt_use_case_1 = (
    "Je vais à Rome la semaine prochaine pour 4 jours (du jeudi au dimanche), arrivée le matin, départ le soir. "
    "Fais un plan de 4 jours avec un budget de 300 EUR pour les sorties et restaurants. "
    "Évite les zones trop touristiques et privilégie les lieux confidentiels."
)

In [5]:
system_prompt_extended = base_system_prompt + """
### tool_get_current_date
- Utiliser cet outil en premier si la demande contient des dates relatives (semaine prochaine, ce week-end, etc.)
- Convertir les dates relatives en dates calendrier explicites avant tout appel dépendant des dates

### tool_geocode_location
- Utiliser cet outil quand l'utilisateur fournit un lieu texte (ville, adresse, quartier)
- Réutiliser les coordonnées retournées pour météo et nearby

### tool_get_weather
- Utiliser cet outil avec coordonnées + dates ISO précises
- Expliquer l'impact météo sur l'ordre des activités et la faisabilité budget

### tool_retrieve_docs
- Utiliser des requêtes ciblées pour récupérer des recommandations locales
- Si la première récupération est faible, reformuler puis relancer
- Ne garder que les preuves réellement utiles à la réponse finale

"""

tools_use_case_1 = [tool_get_current_date, tool_geocode_location, tool_get_weather, tool_retrieve_docs]

agent_use_case_1 = Agent(
    model=model,
    instructions=system_prompt_extended,
    model_settings=settings,
    tools=tools_use_case_1,
)

result_use_case_1 = await run_agent_realtime_logging(
    agent=agent_use_case_1,
    prompt=prompt_use_case_1,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_1.log",
    max_steps=12,
)


RUN TRACE
- prompt : Je vais à Rome la semaine prochaine pour 4 jours (du jeudi au dimanche), arrivée le matin, départ le soir. Fais un plan de 4 jours avec un budget de 300 EUR pour les sorties et restaurants. Évite les zones trop touristiques et privilégie les lieux confidentiels.
STEP 1 : TOOL CALL
- tool : tool_get_current_date
- args :
{}
--------------------------------------------------------------------------------
STEP 2 : TOOL RESULT
- tool : tool_get_current_date
- output :
{
  "current_date": "2026-03-18",
  "weekday_name": "Wednesday"
}
--------------------------------------------------------------------------------
STEP 3 : TOOL CALL
- tool : tool_geocode_location
- args :
{
  "query": "Rome"
}
--------------------------------------------------------------------------------
STEP 4 : TOOL RESULT
- tool : tool_geocode_location
- output :
{
  "name": "Rome, Metropolitan City of Rome Capital, Italy",
  "latitude": 41.8967068,
  "longitude": 12.4822025
}
----------------------

In [6]:
print(result_use_case_1.output)


Voici une proposition de plan pour votre séjour à Rome, axée sur des lieux authentiques et confidentiels, tout en respectant votre budget.

**Résumé**

Ce plan de 4 jours à Rome vous invite à découvrir des quartiers moins fréquentés, des marchés locaux et des trattorias authentiques, loin des foules. Vous explorerez Campo dei Fiori, le quartier de Monti, Testaccio et la colline de l'Aventin, en profitant de la gastronomie romaine traditionnelle et des nouvelles adresses prometteuses. Le budget de 300 EUR est réparti sur les repas et les activités.

**Plan**

*   **Jour 1 (Jeudi 26 mars) : Arrivée et Charme de Campo dei Fiori**
    *   Matin : Arrivée à Rome. Installation. Direction le marché de Campo dei Fiori pour une immersion dans l'ambiance locale et découvrir les produits frais.
    *   Midi : Dégustation de spécialités romaines dans une trattoria traditionnelle près de Campo dei Fiori ou dans le quartier juif, connu pour sa cuisine authentique.
    *   Après-midi : Exploration du

---

## 2. Cas d'usage 2 - Meilleure période Paris -> New York

Travailler un raisonnement long horizon (plusieurs mois).
`tool_get_weather` ne sert que pour le court terme; la preuve principale doit venir de `web_search` et `web_extract`.
La recommandation finale doit comparer des périodes candidates avec compromis explicites (prix, météo, durée de trajet) et sources citées.

In [7]:
prompt_use_case_2 = (
    "Trouve la meilleure période dans les 6 prochains mois pour un voyage Paris -> New York. "
    "Compare météo et informations web sur les prix saisonniers, et justifie la recommandation."
    "Précise les prix, le temps de trajet et les conditions météo."
)

In [8]:
system_prompt_extended = base_system_prompt + """

### tool_get_weather
- Outil réservé aux prévisions court terme
- Ne pas l'utiliser pour prédire la météo à 6 mois ou un climat de saison
- Pour un choix de période long terme, utiliser `web_search` et `web_extract` comme preuves principales
- Utiliser la météo seulement comme signal complémentaire si les dates sont dans la fenêtre supportée

### web_search
- Utiliser cet outil pour collecter des sources publiques récentes sur période, coûts et tendances
- Construire des requêtes ciblées (villes, mois, saison, budget)

### web_extract
- Utiliser cet outil sur les URL les plus prometteuses trouvées via `web_search`
- Extraire uniquement les pages utiles, pas tous les résultats

"""

tools_use_case_2 = tools_use_case_1 + [web_search, web_extract]

agent_use_case_2 = Agent(
    model=model,
    instructions=system_prompt_extended,
    model_settings=settings,
    tools=tools_use_case_2,
)

result_use_case_2 = await run_agent_realtime_logging(
    agent=agent_use_case_2,
    prompt=prompt_use_case_2,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_2.log",
    max_steps=12,
)

RUN TRACE
- prompt : Trouve la meilleure période dans les 6 prochains mois pour un voyage Paris -> New York. Compare météo et informations web sur les prix saisonniers, et justifie la recommandation.Précise les prix, le temps de trajet et les conditions météo.
STEP 1 : TOOL CALL
- tool : tool_get_current_date
- args :
{}
--------------------------------------------------------------------------------
STEP 2 : TOOL RESULT
- tool : tool_get_current_date
- output :
{
  "current_date": "2026-03-18",
  "weekday_name": "Wednesday"
}
--------------------------------------------------------------------------------
STEP 3 : TOOL CALL
- tool : tool_geocode_location
- args :
{
  "query": "New York"
}
--------------------------------------------------------------------------------
STEP 4 : TOOL RESULT
- tool : tool_geocode_location
- output :
{
  "name": "New York, NY, USA",
  "latitude": 40.7127753,
  "longitude": -74.0059728
}
---------------------------------------------------------------------

In [9]:
print(result_use_case_2.output)

Voici une recommandation pour votre voyage de Paris à New York dans les 6 prochains mois.

**Résumé**
La meilleure période pour voyager de Paris à New York dans les 6 prochains mois (mars à septembre 2026) semble être **septembre**. Cette période offre généralement des prix de vols plus bas et des températures confortables, bien que les prévisions de précipitations soient contradictoires.

**Plan**
Il est recommandé de planifier votre voyage en **septembre 2026**.

**Budget**
*   **Vols Paris -> New York :** Les prix des vols sont généralement plus bas en septembre, se situant entre 400 $ et 500 $ [web_search:google.com]. En avril, les prix varient entre 431 $ et 621 $ [web_search:kayak.com, farecompare.com].
*   **Hébergement :** Les informations sur les prix saisonniers de l'hébergement à New York ne sont pas disponibles.

**Météo et Conditions**
*   **Temps de trajet :** Le vol direct de Paris à New York dure environ 7 heures et 46 minutes [web_search:travelmath.com].
*   **Météo en

---

## 3. Cas d'usage 3 - Recommandations proches depuis une adresse


**TODO — Recommandations locales depuis une adresse**

Fichiers à modifier :
- `TP3_travel_planner_Agent/3_1_tooling_assistant.ipynb`
- `shared/agent_tools.py`

Ce cas d'usage combine géolocalisation, recherche de lieux proches et contrainte de budget

Note : modifier `shared/agent_tools.py` uniquement si une fonction outil est incomplète
Bloc de code qui prend une adresse, récupère des lieux proches, applique les contraintes de budget et retourne une sélection courte avec justification


In [10]:
# TODO : construire un prompt avec contraintes budget + adresse explicite
prompt_use_case_3 = (
    "Recommande les meilleurs restaurants et activités près de cette adresse : "
    "10 Rue de la Paix, 75002 Paris, France. "
    "J'ai un budget de 30 euros pour un repas et 20 euros pour une activité. "
)

In [11]:
system_prompt_extended = base_system_prompt + """

### tool_geocode_location
- Utiliser cet outil en premier pour convertir l'adresse texte en coordonnées
- Vérifier la cohérence du lieu avant de lancer nearby

### tool_search_nearby
- Choisir un `place_type` aligné avec l'intention (restaurant, museum, park, etc.)
- Ajouter un `keyword` quand un filtrage fin est nécessaire (calme, local, rooftop, etc.)
- Garder une sélection finale strictement alignée avec les contraintes utilisateur

"""

tools_use_case_3 = tools_use_case_2 + [tool_search_nearby]

agent_use_case_3 = Agent(
    model=model,
    instructions=system_prompt_extended,
    model_settings=settings,
    tools=tools_use_case_3,
)

result_use_case_3 = await run_agent_realtime_logging(
    agent=agent_use_case_3,
    prompt=prompt_use_case_3,
    log_path=ROOT_DIR / "TP3_travel_planner_Agent" / "logs" / "trace_use_case_3.log",
    max_steps=12,
)

RUN TRACE
- prompt : Recommande les meilleurs restaurants et activités près de cette adresse : 10 Rue de la Paix, 75002 Paris, France. J'ai un budget de 30 euros pour un repas et 20 euros pour une activité. 
STEP 1 : TOOL CALL
- tool : tool_geocode_location
- args :
{
  "query": "10 Rue de la Paix, 75002 Paris, France"
}
--------------------------------------------------------------------------------
STEP 2 : TOOL RESULT
- tool : tool_geocode_location
- output :
{
  "name": "10 Rue de la Paix, 75002 Paris, France",
  "latitude": 48.8689443,
  "longitude": 2.3312533
}
--------------------------------------------------------------------------------
STEP 3 : TOOL CALL
- tool : tool_search_nearby
- args :
{
  "latitude": 48.8689443,
  "radius_m": 1500,
  "limit": 5,
  "longitude": 2.3312533,
  "keyword": "bistro",
  "place_type": "restaurant"
}
--------------------------------------------------------------------------------
STEP 4 : TOOL CALL
- tool : tool_search_nearby
- args :
{
  "keywo

In [12]:
print(result_use_case_3.output)

Voici des recommandations pour des restaurants et activités près du 10 Rue de la Paix, Paris, en tenant compte de votre budget.

**Résumé**
Je vous recommande le restaurant o Bistrot de Tom pour un repas abordable et le Musée Rodin ou le Musée de l'Orangerie pour une activité culturelle dans votre budget.

**Plan**
1.  **Repas :** Dégustez un repas au o Bistrot de Tom, situé à proximité.
2.  **Activité culturelle :** Visitez le Musée Rodin pour découvrir ses sculptures ou le Musée de l'Orangerie pour admirer les Nymphéas de Monet.

**Budget**
*   **Restaurant :** o Bistrot de Tom : environ 14,7 $ (environ 13,6 €) par personne [web_search:menuweb.menu]. Le Bistro de Buci est une autre option avec un prix moyen autour de 30 € [web_search:thefork.com].
*   **Activité :**
    *   Musée Rodin : 12 € (en ligne 13 €) [web_search:parisjetaime.com].
    *   Musée de l'Orangerie : 12,50 € [web_search:musee-orangerie.fr].
    *   Musée d'Orsay : Visite nocturne à 12 € (gratuit le premier dimanche